In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from sentence_transformers import models as s_models
from sentence_transformers  import SentenceTransformer
from custum_evals import UBinarySentenceTransformer

def load_model_with_proper_pooling(model_name, model_info):

    base_model = SentenceTransformer(
        model_info["path"],
        model_kwargs=model_info.get("model_config", {}),
    )
    # This is the default behavior for all other models
    print(f"Loading {model_name} with default pooling...")
    if model_info.get("similarity_fn_name") == "hamming":
        print(f"Using UBinarySentenceTransformer for {model_name}")
        model = UBinarySentenceTransformer(
            modules=base_model._modules.values(),
        )
    else:
        print(f"Using SentenceTransformer for {model_name}")
        model = base_model

    return model

In [3]:
model_info = {
    "indus-sde-st-v0.2_polar-monkey-61_30k": {
        "path": "/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000",
        "color": "#33ff77",
    },
    "indus-sde-st-v0.2-61_30k-ubinary_emb": {
        "path": "/rhome/sawale/indus_traning/sentense_transformers/eval/artifacts/model-6hjbp1bx:v1/checkpoint-30000",
        "color": "#9933aa",
        "similarity_fn_name": "hamming",
    }
}

base_model = load_model_with_proper_pooling("indus-sde-st-v0.2_polar-monkey-61_30k", model_info["indus-sde-st-v0.2_polar-monkey-61_30k"])
post_quant_model = load_model_with_proper_pooling("indus-sde-st-v0.2-61_30k-ubinary_emb", model_info["indus-sde-st-v0.2-61_30k-ubinary_emb"])

Loading indus-sde-st-v0.2_polar-monkey-61_30k with default pooling...
Using SentenceTransformer for indus-sde-st-v0.2_polar-monkey-61_30k
Loading indus-sde-st-v0.2-61_30k-ubinary_emb with default pooling...
Using UBinarySentenceTransformer for indus-sde-st-v0.2-61_30k-ubinary_emb


In [4]:
inputs = [
    "This is a test input.",
    "Another test input for verification.",
    "More examples to ensure robustness."
]


In [5]:
base_emb = base_model.encode(inputs, convert_to_tensor=True)
base_emb.shape

torch.Size([3, 768])

In [6]:
post_quant_emb = post_quant_model.encode(inputs, convert_to_tensor=True)
post_quant_emb.shape

torch.Size([3, 96])

In [7]:
post_quant_emb

tensor([[242, 168, 208, 131, 158,  61, 129,  11,  89,  23, 154, 169, 138, 170,
         210, 182, 121, 175, 148, 109,  49,  88,  13, 115,  81, 155, 241,  28,
         142, 119,  88,  31, 107, 211,  70, 137, 200, 163, 197, 172, 215, 120,
         201,  84, 149, 248, 235, 221, 107, 201, 106, 129,  21,  73,  19, 128,
         127, 104, 206,  69, 149, 189, 214,  32, 103, 152, 150, 234,  87, 105,
          96,  50, 225,  92,  44,  58, 104, 222, 224, 167,  17, 124, 182, 212,
         242, 203,  64,   0, 163, 148, 178,  40, 215, 168, 197, 147],
        [226, 224, 217,  34, 223, 249, 136, 171,  89,  37, 154, 236, 137, 165,
         247, 167,  53, 173, 132, 100, 113,  57, 128, 113, 243, 147, 241,  10,
          94,  99,  92,  93, 202, 129, 199, 174, 136, 178, 237, 188, 151, 124,
         232, 148,  23,  63, 239, 101,  42, 139,  14,  23, 149,  76, 149, 214,
         126,  73,  78,  69, 146, 189,  94,  40,  71,  26,  69, 240,  81, 105,
          32,  52, 225, 142,  46,  48, 252,  94, 228, 196,  6

In [8]:
import numpy as np

# now compute using the sde provided logic
def floats_to_binary_bytes(vec):
    """
    Convert float vector to binary quantized bytes using sign bit
    
    Args:
        vec: Float vector [dim]
        
    Returns:
        List of signed integers representing binary quantized vector
    """
    assert vec.ndim == 1, f"Expected 1D vector, got {vec.ndim}D"
    D = vec.shape[0]
    assert D % 8 == 0, f"Dimension must be multiple of 8 for binary packing, got {D}"

    # 1-bit quantization (sign bit)
    bits = (vec >= 0).astype(np.uint8)

    # Pack 8 bits => 1 byte (MSB first)
    packed = np.packbits(bits, bitorder='big')

    # Convert to 8-bit signed integers [-128, 127]
    packed_signed = packed.view(np.int8)
    
    # Convert to Python list for JSON serialization
    return packed_signed.astype(int).tolist()




In [37]:
import torch
sde_post_quant_emb = [
    floats_to_binary_bytes(_emb) for _emb in base_emb.cpu().numpy()
]

sde_post_quant_emb = torch.tensor(sde_post_quant_emb, dtype=torch.int8)


In [38]:
sde_post_quant_emb

tensor([[ -14,  -88,  -48, -125,  -98,   61, -127,   11,   89,   23, -102,  -87,
         -118,  -86,  -46,  -74,  121,  -81, -108,  109,   49,   88,   13,  115,
           81, -101,  -15,   28, -114,  119,   88,   31,  107,  -45,   70, -119,
          -56,  -93,  -59,  -84,  -41,  120,  -55,   84, -107,   -8,  -21,  -35,
          107,  -55,  106, -127,   21,   73,   19, -128,  127,  104,  -50,   69,
         -107,  -67,  -42,   32,  103, -104, -106,  -22,   87,  105,   96,   50,
          -31,   92,   44,   58,  104,  -34,  -32,  -89,   17,  124,  -74,  -44,
          -14,  -53,   64,    0,  -93, -108,  -78,   40,  -41,  -88,  -59, -109],
        [ -30,  -32,  -39,   34,  -33,   -7, -120,  -85,   89,   37, -102,  -20,
         -119,  -91,   -9,  -89,   53,  -83, -124,  100,  113,   57, -128,  113,
          -13, -109,  -15,   10,   94,   99,   92,   93,  -54, -127,  -57,  -82,
         -120,  -78,  -19,  -68, -105,  124,  -24, -108,   23,   63,  -17,  101,
           42, -117,   14, 

In [39]:
import numpy as np
import torch

def unpack_to_binary(packed_data, input_type='int8', bitorder="big"):
    # Validate input type
    if input_type not in ['int8', 'uint8']:
        raise ValueError("input_type must be 'int8' or 'uint8'")

    # Convert to numpy if torch tensor
    if isinstance(packed_data, torch.Tensor):
        packed_np = packed_data.numpy()
    else:
        packed_np = packed_data
    
    # Convert to uint8 for unpacking (np.unpackbits only works with uint8)
    if input_type == 'int8':
        # Convert signed int8 to uint8 view
        packed_uint8 = packed_np.view(np.uint8)
    else:
        # Already uint8 or convert to uint8
        packed_uint8 = packed_np.astype(np.uint8)
    
    # Handle different dimensions
    if packed_np.ndim == 1:
        # Single vector
        binary_bits = np.unpackbits(packed_uint8, bitorder=bitorder)
    else:
        # Multiple vectors (batch)
        binary_bits = np.unpackbits(packed_uint8, axis=1, bitorder=bitorder)
    
    return binary_bits
    

In [40]:
post_quant_emb

tensor([[242, 168, 208, 131, 158,  61, 129,  11,  89,  23, 154, 169, 138, 170,
         210, 182, 121, 175, 148, 109,  49,  88,  13, 115,  81, 155, 241,  28,
         142, 119,  88,  31, 107, 211,  70, 137, 200, 163, 197, 172, 215, 120,
         201,  84, 149, 248, 235, 221, 107, 201, 106, 129,  21,  73,  19, 128,
         127, 104, 206,  69, 149, 189, 214,  32, 103, 152, 150, 234,  87, 105,
          96,  50, 225,  92,  44,  58, 104, 222, 224, 167,  17, 124, 182, 212,
         242, 203,  64,   0, 163, 148, 178,  40, 215, 168, 197, 147],
        [226, 224, 217,  34, 223, 249, 136, 171,  89,  37, 154, 236, 137, 165,
         247, 167,  53, 173, 132, 100, 113,  57, 128, 113, 243, 147, 241,  10,
          94,  99,  92,  93, 202, 129, 199, 174, 136, 178, 237, 188, 151, 124,
         232, 148,  23,  63, 239, 101,  42, 139,  14,  23, 149,  76, 149, 214,
         126,  73,  78,  69, 146, 189,  94,  40,  71,  26,  69, 240,  81, 105,
          32,  52, 225, 142,  46,  48, 252,  94, 228, 196,  6

In [41]:
sde_emb_binary = unpack_to_binary(sde_post_quant_emb, input_type='int8')
llm_emb_binary = unpack_to_binary(post_quant_emb, input_type='uint8', )

In [42]:
sde_emb_binary

array([[1, 1, 1, ..., 0, 1, 1],
       [1, 1, 1, ..., 0, 1, 0],
       [0, 0, 1, ..., 0, 1, 1]], shape=(3, 768), dtype=uint8)

In [43]:
llm_emb_binary

array([[1, 1, 1, ..., 0, 1, 1],
       [1, 1, 1, ..., 0, 1, 0],
       [0, 0, 1, ..., 0, 1, 1]], shape=(3, 768), dtype=uint8)

In [46]:
np.all(sde_emb_binary == llm_emb_binary)


np.True_

In [3]:
from custum_evals import hamming_similarity_from_distance
import torch
import numpy as np

# Method 1: Create proper 2D uint8 tensors (packed bytes)
a = torch.tensor([[255, 0]], dtype=torch.uint8)    # 1 query, 2 bytes (16 bits)
b = torch.tensor([[255, 128]], dtype=torch.uint8)  # 1 corpus, 2 bytes (16 bits)

result = hamming_similarity_from_distance(a, b)
print(f"Result: {result}")

# Method 2: Pack your binary bits into bytes first
def pack_bits_to_bytes(bits):
    """Convert list of bits [1,0,1,0,1,0,1,0] to packed bytes"""
    # Pad to multiple of 8 if needed
    while len(bits) % 8 != 0:
        bits.append(0)
    
    # Convert to numpy array and pack
    bits_array = np.array(bits, dtype=np.uint8)
    packed = np.packbits(bits_array.reshape(-1, 8), axis=1)
    return torch.from_numpy(packed)

# Your original data (pad to 8 bits)
a_bits = [1, 0, 1, 0, 1, 0, 0, 0]  # padded to 8 bits
b_bits = [1, 1, 0, 0, 1, 0, 0, 0]  # padded to 8 bits

a_packed = pack_bits_to_bytes(a_bits)  # Shape: (1, 1) - 1 query, 1 byte
b_packed = pack_bits_to_bytes(b_bits)  # Shape: (1, 1) - 1 corpus, 1 byte

result = hamming_similarity_from_distance(a_packed, b_packed)
print(f"Result with packed bits: {result}")

Result: tensor([[15.]])
Result with packed bits: tensor([[6.]])


In [2]:
from sentence_transformers import util

In [7]:
from datasets import load_dataset
from huggingface_hub import login
import os

# Load the complete benchmark
dataset = load_dataset("/rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/temp/nasa-sde-IR-benchmark-20251024-v2")


In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['_id', 'text'],
        num_rows: 149553
    })
})

In [9]:
from datasets import load_dataset

# Load the specific JSONL file
corpus_dataset = load_dataset(
    "json", 
    data_files="/rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/temp/nasa-sde-IR-benchmark-20251024-v2/corpus.jsonl"
)

# Access the data
corpus = corpus_dataset["train"]  # JSONL files are loaded as "train" split by default
print(f"Number of documents: {len(corpus)}")
print(f"Columns: {corpus.column_names}")
print(f"First document: {corpus[0]}")

Generating train split: 0 examples [00:00, ? examples/s]

Number of documents: 47698
Columns: ['_id', 'text']
First document: {'_id': '0', 'text': 'Chiu_Wiegand_CCMC_Me..>\n\n2017-04-14 18:58\n\n1.1M'}


In [10]:
from datasets import load_dataset

# Load the queries JSONL file
queries_dataset = load_dataset(
    "json", 
    data_files="/rhome/sawale/indus_traning/sentense_transformers/data/sde_new_dump/temp/nasa-sde-IR-benchmark-20251024-v2/queries.jsonl"
)

# Access the data
queries = queries_dataset["train"]  # JSONL files are loaded as "train" split by default
print(f"Number of queries: {len(queries)}")
print(f"Columns: {queries.column_names}")
print(f"First query: {queries[0]}")

# Convert to dictionary format (similar to your corpus)
queries_dict = {row["_id"]: row["text"] for row in queries}
print(f"\nLoaded {len(queries_dict)} queries as dictionary")
print(f"Sample query ID: {list(queries_dict.keys())[0]}")
print(f"Sample query text: {list(queries_dict.values())[0]}")






Generating train split: 0 examples [00:00, ? examples/s]

Number of queries: 101855
Columns: ['_id', 'text']
First query: {'_id': '0', 'text': "What is the file size of the document 'Chiu_Wiegand_CCMC_Me'?"}

Loaded 101855 queries as dictionary
Sample query ID: 0
Sample query text: What is the file size of the document 'Chiu_Wiegand_CCMC_Me'?


In [17]:
from sentence_transformers import SentenceTransformer, util

# 1. Load a Hugging Face model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", tokenizer_kwargs={"model_max_length": 10, "truncation": True})

# 2. (Optional) Limit max number of tokens
# model.max_seq_length = 1

# 3. Some example sentences
sentences = [
    "I love to hike",
    "Deep models need large datasets.",
    "I enjoy hiking in snow."
]

# 4. Generate embeddings
embeddings = model.encode(
    sentences,
    truncate_dim=256,
    convert_to_tensor=True,   # returns a PyTorch tensor
    show_progress_bar=True,
    # model_max_length=10
)

# 5. Compute similarity between sentence 1 and others
similarities = util.cos_sim(embeddings[0], embeddings[1:])

print("Embedding shape:", embeddings.shape)
print("Cosine similarities:", similarities)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: torch.Size([3, 256])
Cosine similarities: tensor([[0.0394, 0.7485]], device='cuda:0')


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Reduce context window (default is often 256-512)
model.max_seq_length = 64  # or 128, 256, etc.

# Embeddings still have full dimensions (384 for this model)
embeddings = model.encode(["Your text here"])
print(embeddings.shape)  # e.g., (1, 384) - full embedding size preserved

(1, 384)


In [11]:
model = SentenceTransformer('all-MiniLM-L6-v2')
# model.max_seq_length = 64

text = " ".join(["token"] * 200)

# Tokenize directly
encoded = model.tokenizer(
    text,
    padding=True,
    truncation=True,
    max_length=model.max_seq_length,
    return_tensors='pt'
)

print(f"Actual tokens passed to model: {encoded['input_ids'].shape[1]}")
print(f"Model max_seq_length setting: {model.max_seq_length}")
# These should match (or be very close)

Actual tokens passed to model: 202
Model max_seq_length setting: 256


In [6]:
from sentence_transformers import SentenceTransformer, util

# 1. Load a Hugging Face model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", tokenizer_kwargs={"max_length": 128, "truncation": True,})

# 2. (Optional) Limit max number of tokens
# model.max_seq_length = 256

# 3. Some example sentences
sentences = [
    "I love to hike",
    "Deep models need large datasets.",
    "I enjoy hiking in snow."
]

# 4. Generate embeddings
embeddings = model.encode(
    sentences,
    convert_to_tensor=True,   # returns a PyTorch tensor
    show_progress_bar=True
)

# 5. Compute similarity between sentence 1 and others
similarities = util.cos_sim(embeddings[0], embeddings[1:])

print("Embedding shape:", embeddings.shape)
print("Cosine similarities:", similarities)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: torch.Size([3, 384])
Cosine similarities: tensor([[0.0458, 0.7439]], device='cuda:0')


In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
model.max_seq_length = 32

# 1. Create a dummy input clearly longer than 1024 tokens
long_text = ["word " * 2000] 

# 2. Use the model's internal tokenizer
# This function automatically looks at model.max_seq_length
features = model.tokenize(long_text)

# 3. Check the length of the input_ids
actual_length = features['input_ids'].shape[1]

print(f"Set limit: {model.max_seq_length}")
print(f"Actual tokenized length: {actual_length}")

# Verification logic
if actual_length == 32:
    print("✅ Success: The model is truncating at 1024.")
else:
    print(f"❌ Failed: The model is using {actual_length} tokens.")

Set limit: 32
Actual tokenized length: 32
✅ Success: The model is truncating at 1024.


In [2]:
features

{'input_ids': tensor([[ 101, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773,
          2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773, 2773,
          2773, 2773, 2773, 2773, 2773, 2773, 2773,  102]]),
 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
          0, 0, 0, 0, 0, 0, 0, 0]]),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
          1, 1, 1, 1, 1, 1, 1, 1]])}